# Entraînement YOLOv8 — conditions de peau (Phase 2 Lumina)

Ce notebook entraîne le détecteur de conditions de peau et exporte le modèle ONNX attendu par le backend (`SKIN_YOLO_MODEL_PATH`).

**Avant de lancer :**
1. Runtime → *Change runtime type* → **T4 GPU**.
2. Créer un compte sur [universe.roboflow.com](https://universe.roboflow.com) et **fusionner** des datasets (acné, points noirs, taches, pores, rides) dans **un seul projet à 5 classes**, impérativement dans cet ordre (celui de `CLASS_TO_CONDITION` dans `app/skin/local.py`) :

| id | classe |
|----|--------|
| 0 | acne |
| 1 | blackheads |
| 2 | dark_spots |
| 3 | enlarged_pores |
| 4 | wrinkles |

3. Récupérer la clé API Roboflow : *Settings → API Keys*.
4. Renseigner les variables de la cellule Configuration ci-dessous, puis *Runtime → Run all*.

In [ ]:
# Vérification du GPU (doit afficher une Tesla T4)
!nvidia-smi -L

In [ ]:
# Configuration — À REMPLIR
ROBOFLOW_API_KEY = ""   # Settings -> API Keys
WORKSPACE = ""          # slug du workspace Roboflow
PROJECT = ""            # slug du projet fusionné à 5 classes
VERSION = 1              # numéro de version du dataset
EPOCHS = 60
IMGSZ = 640
MODEL = "yolov8n.pt"    # nano : rapide en CPU côté serveur ; yolov8s.pt si le GPU suit

In [ ]:
%pip install -q ultralytics roboflow

In [ ]:
# Téléchargement du dataset au format YOLOv8
from roboflow import Roboflow

dataset = (
    Roboflow(api_key=ROBOFLOW_API_KEY)
    .workspace(WORKSPACE)
    .project(PROJECT)
    .version(VERSION)
    .download("yolov8")
)
DATA_YAML = f"{dataset.location}/data.yaml"

# Garde-fou : l'ordre des classes DOIT correspondre à CLASS_TO_CONDITION du backend.
import yaml
names = yaml.safe_load(open(DATA_YAML))["names"]
print("Classes du dataset :", names)
attendu = ["acne", "blackheads", "dark_spots", "enlarged_pores", "wrinkles"]
assert [n.lower().replace("-", "_") for n in names] == attendu, (
    f"Ordre des classes incorrect ! Attendu {attendu}, obtenu {names}. "
    "Corriger le projet Roboflow avant d'entraîner."
)
print("Ordre des classes OK ✔")

In [ ]:
# Entraînement (~30-60 min sur T4 selon la taille du dataset)
from ultralytics import YOLO

model = YOLO(MODEL)
results = model.train(data=DATA_YAML, epochs=EPOCHS, imgsz=IMGSZ)
BEST = str(results.save_dir / "weights" / "best.pt")
print("Meilleurs poids :", BEST)

In [ ]:
# Export ONNX — nms=True obligatoire (format de sortie attendu par LocalAnalyzer._yolo_scores)
from ultralytics import YOLO

onnx_path = YOLO(BEST).export(format="onnx", nms=True, imgsz=IMGSZ)
print("Modèle ONNX exporté :", onnx_path)

In [ ]:
# Téléchargement du fichier sur ton PC
from google.colab import files

files.download(onnx_path)

## Après le téléchargement

1. Poser le fichier sur le PC, par exemple `backend/models/skin_yolo.onnx`.
2. Dans `backend/.env` :
   ```
   SKIN_YOLO_MODEL_PATH=models/skin_yolo.onnx
   ANALYSIS_ENGINE=auto
   ```
3. Redémarrer le backend — le moteur local complète alors Face++ (repli automatique).